In [64]:
import pandas as pd
import numpy as np
import unicodedata
import re

### Presuntos Suicidios en Colombia 2015 - 2024

Fuente: https://www.datos.gov.co/Justicia-y-Derecho/Presuntos-Suicidios-Colombia-2015-a-2024-Cifras-de/f75u-mirk/about_data

### 1. Extracción

In [65]:
# 1. Extracción 
df = pd.read_csv('../data/Presuntos_Suicidios._Colombia,_2015_a_2024.csv')

print("Forma inicial:", df.shape)
print("\nValores nulos antes de la limpieza:\n", df.isnull().sum())

print("\nTipos de atributos en df:")
print(df.dtypes)

Forma inicial: (26558, 32)

Valores nulos antes de la limpieza:
 ID                                            0
Año del hecho                                 0
Grupo de Edad Quinquenal                      0
Grupo Mayor Menor de Edad                     0
Grupo de Edad judicial                        0
Ciclo Vital                                   0
Sexo de la victima                            0
Estado Civil                                  0
País de Nacimiento                            0
Escolaridad                                   0
Pertenencia Grupal                            0
Pertenencia Étnica                            0
Mes del hecho                                 0
Dia del hecho                                 0
Rango de Hora del Hecho X 3 Horas             0
Código Dane Municipio                         0
Municipio del hecho DANE                      0
Departamento del hecho DANE                   0
Código Dane Departamento                      0
Escenario del Hecho    

### 2. Limpieza

In [66]:
# Mostrar las categorías únicas de cada atributo categórico
for col in cat_cols:
    print(f"\n{'=' * 80}\n{col}")
    print(f"Total de categorías únicas: {df[col].nunique(dropna=False)}")
    print(df[col].unique())


Grupo de Edad Quinquenal 
Total de categorías únicas: 17
<StringArray>
[ '(18 a 19)',  '(25 a 29)',  '(35 a 39)',  '(55 a 59)',  '(45 a 49)',
  '(65 a 69)',  '(30 a 34)',  '(20 a 24)',  '(60 a 64)',  '(40 a 44)',
 '(80 y más)',  '(15 a 17)',  '(50 a 54)',  '(75 a 79)',  '(70 a 74)',
  '(10 a 14)',  '(05 a 09)']
Length: 17, dtype: str

Grupo Mayor Menor de Edad
Total de categorías únicas: 4
<StringArray>
[  'b) Mayor de Edad (>18 Años)',   'a) Menor de Edad (<18 Años)',
 'b) Mayores de Edad (>18 años)', 'a) Menores de Edad (<18 años)']
Length: 4, dtype: str

Grupo de Edad judicial
Total de categorías únicas: 17
<StringArray>
[ '(18 a 19)',  '(25 a 28)',  '(35 a 39)',  '(55 a 59)',  '(45 a 49)',
  '(65 a 69)',  '(29 a 34)',  '(20 a 24)',  '(60 a 64)',  '(40 a 44)',
 '(80 y más)',  '(14 a 17)',  '(50 a 54)',  '(75 a 79)',  '(70 a 74)',
  '(10 a 13)',  '(05 a 09)']
Length: 17, dtype: str

Ciclo Vital
Total de categorías únicas: 5
<StringArray>
[      '(18 a 28) Juventud',        '(29 a 59

In [67]:
def normalizar_categoria(valor):
    if pd.isna(valor):
        return valor
    
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = texto.upper().strip()
    
    # Reducir múltiples espacios a uno solo sin fusionar las palabras
    texto = re.sub(r'\s+', ' ', texto)
    
    # Ajustar espacios alrededor de los paréntesis
    texto = texto.replace(" (", "(").replace("( ", "(").replace(" )", ")")
    
    return texto

for col in cat_cols:
    df[col] = df[col].map(normalizar_categoria)

In [68]:
# Mostrar las categorías únicas de cada atributo categórico después de la estandarización
for col in cat_cols:
    print(f"\n{'=' * 80}\n{col}")
    print(f"Total de categorías únicas: {df[col].nunique(dropna=False)}")
    print(df[col].unique())


Grupo de Edad Quinquenal 
Total de categorías únicas: 17
<StringArray>
[ '(18 A 19)',  '(25 A 29)',  '(35 A 39)',  '(55 A 59)',  '(45 A 49)',
  '(65 A 69)',  '(30 A 34)',  '(20 A 24)',  '(60 A 64)',  '(40 A 44)',
 '(80 Y MAS)',  '(15 A 17)',  '(50 A 54)',  '(75 A 79)',  '(70 A 74)',
  '(10 A 14)',  '(05 A 09)']
Length: 17, dtype: str

Grupo Mayor Menor de Edad
Total de categorías únicas: 4
<StringArray>
[  'B) MAYOR DE EDAD(>18 ANOS)',   'A) MENOR DE EDAD(<18 ANOS)',
 'B) MAYORES DE EDAD(>18 ANOS)', 'A) MENORES DE EDAD(<18 ANOS)']
Length: 4, dtype: str

Grupo de Edad judicial
Total de categorías únicas: 17
<StringArray>
[ '(18 A 19)',  '(25 A 28)',  '(35 A 39)',  '(55 A 59)',  '(45 A 49)',
  '(65 A 69)',  '(29 A 34)',  '(20 A 24)',  '(60 A 64)',  '(40 A 44)',
 '(80 Y MAS)',  '(14 A 17)',  '(50 A 54)',  '(75 A 79)',  '(70 A 74)',
  '(10 A 13)',  '(05 A 09)']
Length: 17, dtype: str

Ciclo Vital
Total de categorías únicas: 5
<StringArray>
[      '(18 A 28) JUVENTUD',        '(29 A 59) AD

In [69]:
from difflib import SequenceMatcher

# Detectar categorías potencialmente similares

for col in cat_cols:
    categorias = (
        df[col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    similares = []

    for i, cat_1 in enumerate(categorias):
        for cat_2 in categorias[i + 1:]:
            similitud = SequenceMatcher(
                None, cat_1, cat_2
            ).ratio()

            palabras_1 = set(cat_1.split())
            palabras_2 = set(cat_2.split())
            comun = palabras_1 & palabras_2

            if similitud >= 0.90 or len(comun) >= 6:
                similares.append((cat_1, cat_2, round(similitud, 2)))

    if similares:
        print(f"\n{'=' * 80}\n{col}")
        for categoria_1, categoria_2, similitud in similares:
            print(f"{similitud}: {categoria_1} <--> {categoria_2}")


# Homologaciones semánticas
homologaciones = {
    'Escolaridad': {
        'EDUCACION BASICA PRIMARIA': 'BASICA PRIMARIA',
        'EDUCACION BASICA SECUNDARIA': 'BASICA SECUNDARIA',
        'EDUCACION MEDIA O SECUNDARIA ALTA': 'MEDIA VOCACIONAL',
        'ESPECIALIZACION MAESTRIA O EQUIVALENTE': 'POSGRADO',
        'MAESTRIA': 'POSGRADO',
        'ESPECIALIZACION': 'POSGRADO',
        'DOCTORADO': 'POSGRADO',
    },
    'Pertenencia Étnica': {
        'SIN INFORMACION': 'NO REGISTRA',
        'SIN PERTENENCIA ETNICA': 'NINGUNA',
    },
    'Pertenencia Grupal': {
        'GRUPOS ETNICOS': 'PERTENECIENTES A GRUPOS ETNICOS',
        'CAMPESINOS(AS) Y/O TRABAJADORES(AS) DEL CAMPO':
            'CAMPESINOS (AS) Y/O TRABAJADORES (AS) DEL CAMPO',
    },
    'Zona del Hecho': {
        'PARTE RURAL(VEREDA Y CAMPO)':
            'PARTE RURAL (VEREDA Y CAMPO)',
    },
    'Escenario del Hecho': {
        'CENTRO POBLADO(CORREGIMIENTO, INSPECCION DE POLICIA)':
            'CENTRO POBLADO (CORREGIMIENTO, INSPECCION DE POLICIA)',
    }
}

for columna, mapa in homologaciones.items():
    if columna in df.columns:
        df[columna] = df[columna].replace(mapa)


# Verificar las categorías finales
for columna in homologaciones:
    if columna in df.columns:
        print(f"\n{columna}:")
        print(df[columna].value_counts(dropna=False))


Grupo Mayor Menor de Edad
0.96: B) MAYOR DE EDAD(>18 ANOS) <--> B) MAYORES DE EDAD(>18 ANOS)
0.96: A) MENOR DE EDAD(<18 ANOS) <--> A) MENORES DE EDAD(<18 ANOS)

Pertenencia Grupal
0.94: MAESTRO - EDUCADOR <--> MAESTRO / EDUCADOR
0.87: EJERCICIO DE ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO <--> PERSONAS QUE EJERCEN ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO
0.88: EJERCICIO DE ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO <--> PERSONA QUE EJERCE ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO
0.95: RELIGIOSOS <--> RELIGIOSO
0.99: PERSONAS QUE EJERCEN ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO <--> PERSONA QUE EJERCE ACTIVIDADES RELACIONADAS CON LA SALUD EN ZONAS DE CONFLICTO
0.94: MAESTRO/EDUCADOR <--> MAESTRO / EDUCADOR
0.97: COMUNIDAD LGBTI <--> COMUNIDAD LGBT
0.91: RECICLADORES <--> RECICLADOR
0.98: PERSONAS QUE EJERCEN ACTIVIDADES POLITICAS <--> PERSONA QUE EJERCE ACTIVIDADES POLITICAS
0.98: PERSONAS QUE E

In [70]:
# Transformación y estandarización final del DataFrame
df = df.copy()

# Normalizar nombres de columnas
df.columns = df.columns.str.strip()

# Normalizar valores de texto
cols_texto = df.select_dtypes(include=["object", "string"]).columns

for col in cols_texto:
    df[col] = df[col].map(normalizar_categoria)

# Reemplazar valores vacíos por NaN
valores_vacios = ["", "N/A", "NA", "NAN", "NULL", "NONE"]

for col in cols_texto:
    df[col] = df[col].replace(valores_vacios, np.nan)

# Aplicar homologaciones semánticas
for columna, mapa in homologaciones.items():
    if columna in df.columns:
        df[columna] = df[columna].replace(mapa)

# Correcciones adicionales de categorías
correcciones = {
    "Grupo Mayor Menor de Edad": {
        "B) MAYORES DE EDAD(>18 ANOS)": "B) MAYOR DE EDAD(>18 ANOS)"
    }
}

for columna, mapa in correcciones.items():
    if columna in df.columns:
        df[columna] = df[columna].replace(mapa)

# Convertir variables numéricas al tipo adecuado
for col in cols_enteras:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Eliminar registros completamente duplicados
df = df.drop_duplicates().reset_index(drop=True)

# Eliminar duplicados por ID, conservando el primer registro
if "ID" in df.columns:
    df = df.drop_duplicates(subset="ID", keep="first").reset_index(drop=True)

# Verificación de calidad
print("Forma final:", df.shape)
print("\nValores nulos:")
print(df.isnull().sum())

print("\nTipos de datos:")
print(df.dtypes)

print("\nRegistros duplicados:", df.duplicated().sum())

Forma final: (26558, 32)

Valores nulos:
ID                                            0
Año del hecho                                 0
Grupo de Edad Quinquenal                      0
Grupo Mayor Menor de Edad                     0
Grupo de Edad judicial                        0
Ciclo Vital                                   0
Sexo de la victima                            0
Estado Civil                                  0
País de Nacimiento                            0
Escolaridad                                   0
Pertenencia Grupal                            0
Pertenencia Étnica                            0
Mes del hecho                                 0
Dia del hecho                                 0
Rango de Hora del Hecho X 3 Horas             0
Código Dane Municipio                         0
Municipio del hecho DANE                      0
Departamento del hecho DANE                   0
Código Dane Departamento                      0
Escenario del Hecho                           0

In [71]:
# Imprimir los nombres de las columnas del DataFrame
for col in df.columns:
    print(col)

ID
Año del hecho
Grupo de Edad Quinquenal
Grupo Mayor Menor de Edad
Grupo de Edad judicial
Ciclo Vital
Sexo de la victima
Estado Civil
País de Nacimiento
Escolaridad
Pertenencia Grupal
Pertenencia Étnica
Mes del hecho
Dia del hecho
Rango de Hora del Hecho X 3 Horas
Código Dane Municipio
Municipio del hecho DANE
Departamento del hecho DANE
Código Dane Departamento
Escenario del Hecho
Zona del Hecho
Circunstancia del Hecho Detallada
Manera de Muerte
Mecanismo Causal de la Lesión Fatal
Diagnostico Topográfico de la Lesión Fatal
Razón del Suicidio
Localidad del Hecho
Ancestro Racial
Pueblo Indígena
Orientación Sexual
Identidad de Género
Transgénero


### Contrucción de Dimensiones y Hechos


Dimensiones

In [72]:
# 1. Dimensión Tiempo
dim_tiempo = df[['Año del hecho', 'Mes del hecho', 'Dia del hecho', 'Rango de Hora del Hecho X 3 Horas']].drop_duplicates().reset_index(drop=True)
dim_tiempo.index += 1
dim_tiempo = dim_tiempo.reset_index().rename(columns={'index': 'id_tiempo'})

# 2. Dimensión Geografía
dim_geografia = df[['Código Dane Municipio', 'Municipio del hecho DANE', 'Código Dane Departamento', 'Departamento del hecho DANE', 'Zona del Hecho', 'Localidad del Hecho']].drop_duplicates().reset_index(drop=True)
dim_geografia.index += 1
dim_geografia = dim_geografia.reset_index().rename(columns={'index': 'id_geografia'})

# 3. Dimensión Víctima
cols_victima = ['Sexo de la victima', 'Grupo de Edad Quinquenal', 'Grupo Mayor Menor de Edad', 'Grupo de Edad judicial', 'Ciclo Vital', 'Estado Civil', 'País de Nacimiento', 'Escolaridad', 'Pertenencia Grupal', 'Pertenencia Étnica', 'Ancestro Racial', 'Pueblo Indígena', 'Orientación Sexual', 'Identidad de Género', 'Transgénero']
dim_victima = df[cols_victima].drop_duplicates().reset_index(drop=True)
dim_victima.index += 1
dim_victima = dim_victima.reset_index().rename(columns={'index': 'id_victima'})

# 4. Dimensión Circunstancia
cols_circunstancia = ['Escenario del Hecho', 'Circunstancia del Hecho Detallada', 'Manera de Muerte', 'Mecanismo Causal de la Lesión Fatal', 'Diagnostico Topográfico de la Lesión Fatal', 'Razón del Suicidio']
dim_circunstancia = df[cols_circunstancia].drop_duplicates().reset_index(drop=True)
dim_circunstancia.index += 1
dim_circunstancia = dim_circunstancia.reset_index().rename(columns={'index': 'id_circunstancia'})

Hechos

In [75]:
# Unificar IDs dimensionales al DataFrame principal
df_fact = df.merge(dim_tiempo, on=['Año del hecho', 'Mes del hecho', 'Dia del hecho', 'Rango de Hora del Hecho X 3 Horas'], how='left')
df_fact = df_fact.merge(dim_geografia, on=['Código Dane Municipio', 'Municipio del hecho DANE', 'Código Dane Departamento', 'Departamento del hecho DANE', 'Zona del Hecho', 'Localidad del Hecho'], how='left')
df_fact = df_fact.merge(dim_victima, on=cols_victima, how='left')
df_fact = df_fact.merge(dim_circunstancia, on=cols_circunstancia, how='left')

# Seleccionar solo las llaves foráneas y crear la métrica de agregación
fact_suicidios = df_fact[['ID', 'id_tiempo', 'id_geografia', 'id_victima', 'id_circunstancia']].copy()
fact_suicidios.rename(columns={'ID': 'id_hecho_original'}, inplace=True)
fact_suicidios['cantidad'] = 1